
# 01 · Exploración de esquemas — Ingresantes, Matriculados e Institucional

**Objetivo:** diagnóstico reproducible de la estructura y calidad de los datasets *raw* de TUNI.pe para decidir si los archivos de cada grupo son **concatenables** y qué tratamiento requiere cada fuente.

**Reglas del notebook:**
- **Solo diagnóstico**: no concatena archivos, no modifica los CSVs originales.
- **Detección dinámica**: las columnas, dtypes y granularidad se detectan a partir de los datos (cardinalidad y nombres), **sin listas hardcodeadas**.
- **Contrato para el pipeline**: el resultado de esquema de cada dataset se guarda en `data/schemas/<dataset>_schema.json`. `etl_pipeline.py` leerá ese JSON para saber qué columnas procesar.
- **No se inventan claves ni significados**: lo que no pueda determinarse con los datos queda marcado como *pendiente de validación*.


In [1]:

import pandas as pd
import numpy as np
import gc
import re
import json
from pathlib import Path
from collections import Counter
from IPython.display import display, Markdown

print("pandas", pd.__version__)


pandas 3.0.3


In [2]:

from pathlib import Path


# ============================================================
# DETECCIÓN AUTOMÁTICA DE LA RAÍZ DEL PROYECTO
# ============================================================
# El notebook puede ejecutarse desde:
#   1. La carpeta raíz del proyecto (donde está data/)
#   2. La carpeta notebooks/ (subcarpeta del proyecto)
#
# Esta lógica sube un nivel si es necesario para encontrar 'data/'.
# NO USAR Path("data") DIRECTO porque falla si el directorio de trabajo no es la raíz.
# ============================================================


current_dir = Path.cwd()


if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta 'data'. Asegúrate de que el notebook "
        "esté en la raíz del proyecto o dentro de la carpeta 'notebooks/'."
    )


print(f"📁 Raíz del proyecto detectada: {PROJECT_ROOT}")


# ============================================================
# CONFIGURACIÓN DE RUTAS (ARQUITECTURA MEDALLION)
# ============================================================
DATA = PROJECT_ROOT / "data"
RAW = DATA / "Bronce"
SCHEMAS = DATA / "schemas"


ING_DIR = RAW / "ingresantes_raw"
MAT_DIR = RAW / "matriculados_raw"
INST_FILE = RAW / "institucional.csv"


# Crear la carpeta de schemas si no existe (no afecta los datos raw)
SCHEMAS.mkdir(parents=True, exist_ok=True)


# Verificación de existencia
print("\n--- Verificación de estructura ---")
for p in (RAW, ING_DIR, MAT_DIR):
    print(f"{p}: {'✅ EXISTE' if p.exists() else '❌ NO EXISTE'}")
print(f"{INST_FILE}: {'✅ EXISTE' if INST_FILE.exists() else '❌ NO EXISTE'}")
print(f"{SCHEMAS}: {'✅ CREADA' if SCHEMAS.exists() else '❌ ERROR'}")
print("-" * 50)


📁 Raíz del proyecto detectada: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP

--- Verificación de estructura ---
/mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Bronce: ✅ EXISTE
/mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Bronce/ingresantes_raw: ✅ EXISTE
/mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Bronce/matriculados_raw: ✅ EXISTE
/mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Bronce/institucional.csv: ✅ EXISTE
/mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/schemas: ✅ CREADA
--------------------------------------------------


In [3]:

# Convenciones detectadas en los CSVs de TUNI.pe:
SEP = "|"            # ingresantes y matriculados usan pipe '|' como separador
ENCODING = "latin-1" # los textos contienen tildes, ñ, etc. (ISO-8859-1)

# institucional.csv usa coma y también encoding latin-1 (con comas internas no escapadas).
INST_SEP = ","
INST_ENCODING = "latin-1"


In [4]:

def load_csv(path, sep=SEP, encoding=ENCODING, nrows=None, **kwargs):
    """Carga un CSV de TUNI.pe con las convenciones por defecto."""
    return pd.read_csv(path, sep=sep, encoding=encoding,
                       nrows=nrows, low_memory=False, **kwargs)


def inspect_csv(path, sep=SEP, encoding=ENCODING, nrows=None):
    """Inspección completa de un CSV en un solo pase (dinámica, sin columnas fijas):
    dimensiones, columnas, dtypes, nulos, cardinalidad y duplicados exactos."""
    df = load_csv(path, sep=sep, encoding=encoding, nrows=nrows)
    info = {
        "archivo": Path(path).name,
        "n_filas": len(df),
        "n_columnas": len(df.columns),
        "columnas": list(df.columns),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "nulos": df.isna().sum().to_dict(),
        "pct_nulos": (df.isna().mean() * 100).round(2).to_dict(),
        "cardinalidad": df.nunique(dropna=False).to_dict(),
        "duplicados_exactos": int(df.duplicated().sum()),
    }
    return df, info


def tabla_resumen(reports):
    """Convierte un dict de reportes (archivo -> info) en una tabla plana."""
    return pd.DataFrame([
        {"archivo": r["archivo"], "n_filas": r["n_filas"],
         "n_columnas": r["n_columnas"],
         "duplicados_exactos": r["duplicados_exactos"]}
        for r in reports.values()
    ])


In [5]:

def detect_identifier_candidates(df, max_candidates=8):
    """Detecta dinámicamente columnas candidatas a identificadores/claves.
    Combina heurística de nombre (GUID/CODIGO/ID) con cardinalidad alta.
    NO usa listas fijas de columnas."""
    n = len(df)
    ratio = df.nunique(dropna=False) / n
    por_nombre = [c for c in df.columns
                  if any(k in c.upper() for k in ("GUID", "CODIGO", "_ID"))]
    por_cardinalidad = [c for c in ratio.index
                        if c not in por_nombre and ratio[c] >= 0.3]
    pool = por_nombre + sorted(por_cardinalidad, key=lambda c: -ratio[c])
    return pool[:max_candidates]


def find_greedy_key(df, candidates, max_columns=6):
    """Selección voraz de clave candidata por cardinalidad.
    Añade columnas (de mayor a menor cardinalidad) hasta lograr 0 duplicados
    o agotar el pool. Devuelve (clave, duplicados_restantes).
    Es una HIPÓTESIS por cardinalidad, no una clave de negocio confirmada."""
    clave = []
    for col in candidates[:max_columns]:
        clave.append(col)
        dups = int(df.duplicated(subset=clave).sum())
        if dups == 0:
            return clave, dups
    return clave, int(df.duplicated(subset=clave).sum())


def detect_period_col(df):
    """Detecta la columna de periodo académico estandarizado por patrón de nombre."""
    match = [c for c in df.columns if "ESTANDAR" in c.upper()]
    return match[0] if match else None


def build_schema(dataset, dir_path, sep, encoding, reportes,
                 clave_candidata, dups_clave, period_col):
    """Ensambla el contrato de esquema (dict) que leerá etl_pipeline.py."""
    primero = list(reportes.values())[0]
    return {
        "dataset": dataset,
        "fuente": str(dir_path),
        "n_archivos": len(reportes),
        "n_filas_total": sum(r["n_filas"] for r in reportes.values()),
        "encoding": encoding,
        "separador": sep,
        "columna_periodo": period_col,
        "columnas": primero["columnas"],
        "dtypes": primero["dtypes"],
        "granularidad": {
            "metodo": "hipótesis por cardinalidad (selección voraz)",
            "clave_candidata": clave_candidata,
            "duplicados_con_clave": dups_clave,
            "nivel_confianza": "pendiente de validación de negocio",
        },
        "calidad": {
            "duplicados_exactos_total": sum(r["duplicados_exactos"]
                                            for r in reportes.values()),
            "nulos_por_columna": {
                c: int(sum(r["nulos"][c] for r in reportes.values()))
                for c in primero["columnas"]
            },
        },
    }


def save_schema(schema, path):
    """Guarda el esquema como JSON (contrato para el pipeline)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(schema, fh, ensure_ascii=False, indent=2)
    print(f"Esquema guardado: {path}")
    return path



# PARTE 1 — INGRESANTES

Inventario, inspección y **detección dinámica** de columnas y granularidad. Cada archivo se carga completo y secuencialmente; el DataFrame se descarta tras la inspección.


In [6]:

ING_FILES = sorted(ING_DIR.glob("*.csv"))
print(f"Archivos encontrados: {len(ING_FILES)}")
for f in ING_FILES:
    print(f"  {f.name:28s} {f.stat().st_size/1e6:9.1f} MB")


Archivos encontrados: 6
  ingresante_2020.csv              142.3 MB
  ingresante_2021.csv              158.0 MB
  ingresante_2022.csv              160.5 MB
  ingresante_2023.csv              175.4 MB
  ingresante_2024.csv              183.9 MB
  ingresante_2025.csv              197.2 MB


In [7]:

ing_reportes = {}
ing_granularidad = {}
period_col_ing = None

for f in ING_FILES:
    df, info = inspect_csv(f)
    if period_col_ing is None:
        period_col_ing = detect_period_col(df)
    cand = detect_identifier_candidates(df)
    clave, dups = find_greedy_key(df, cand)
    ing_granularidad[f.name] = {
        "candidatos": cand, "clave": clave, "duplicados": dups,
    }
    ing_reportes[f.name] = info
    del df
    gc.collect()

print("Inspección completada. Reportes en 'ing_reportes'.")
print(f"Columna de periodo detectada: {period_col_ing}")


Inspección completada. Reportes en 'ing_reportes'.
Columna de periodo detectada: PROCESO_ESTANDARIZADO


In [8]:

ing_summary = tabla_resumen(ing_reportes)
display(ing_summary)
print(f"Total filas ingresantes: {ing_summary['n_filas'].sum():,}")

print("\nGranularidad detectada por archivo (hipótesis por cardinalidad):")
g = pd.DataFrame(ing_granularidad).T
g["candidatos"] = g["candidatos"].apply(lambda x: ", ".join(x))
g["clave"] = g["clave"].apply(lambda x: " + ".join(x))
display(g[["candidatos", "clave", "duplicados"]])


,archivo,n_filas,n_columnas,duplicados_exactos
0,ingresante_2020.csv,434038,32,0
1,ingresante_2021.csv,483029,32,0
2,ingresante_2022.csv,490593,32,0
3,ingresante_2023.csv,536895,32,0
4,ingresante_2024.csv,565860,32,0
5,ingresante_2025.csv,609872,32,0


Total filas ingresantes: 3,120,287

Granularidad detectada por archivo (hipótesis por cardinalidad):


,candidatos,clave,duplicados
ingresante_2020.csv,"CODIGO_INEI, GUID_PERSONA, CODIGO_SIU_PROGRAMA...",CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRA...,1
ingresante_2021.csv,"CODIGO_INEI, GUID_PERSONA, CODIGO_SIU_PROGRAMA...",CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRA...,3
ingresante_2022.csv,"CODIGO_INEI, GUID_PERSONA, CODIGO_SIU_PROGRAMA...",CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRA...,1
ingresante_2023.csv,"CODIGO_INEI, GUID_PERSONA, CODIGO_SIU_PROGRAMA...",CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRA...,1
ingresante_2024.csv,"CODIGO_INEI, GUID_PERSONA, CODIGO_SIU_PROGRAMA...",CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA,0
ingresante_2025.csv,"CODIGO_INEI, GUID_PERSONA, CODIGO_SIU_PROGRAMA...",CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA,0


In [9]:

base = list(ing_reportes.values())[0]
print("--- Comparación de columnas ---")
same_cols = all(r["columnas"] == base["columnas"] for r in ing_reportes.values())
print(f"¿Mismas columnas en los {len(ing_reportes)} archivos? -> {same_cols}")
if same_cols:
    print(f"Columnas ({len(base['columnas'])}):")
    for i, c in enumerate(base["columnas"], 1):
        print(f"  {i:2d}. {c}")

print("\n--- Comparación de dtypes ---")
same_dtypes = all(r["dtypes"] == base["dtypes"] for r in ing_reportes.values())
print(f"¿Mismos dtypes en los {len(ing_reportes)} archivos? -> {same_dtypes}")
if not same_dtypes:
    diffs = {}
    for name, r in ing_reportes.items():
        for c, dt in r["dtypes"].items():
            diffs.setdefault(c, set()).add(dt)
    print("Columnas con dtype inconsistente:",
          {c: sorted(v) for c, v in diffs.items() if len(v) > 1})


--- Comparación de columnas ---
¿Mismas columnas en los 6 archivos? -> True
Columnas (32):
   1. CODIGO_INEI
   2. NOMBRE_ENTIDAD
   3. TIPO_ENTIDAD
   4. TIPO_GESTION
   5. LICENCIADO
   6. TIPO_CONSTITUCION
   7. NIVEL_ACADEMICO
   8. PROCESO_ESTANDARIZADO
   9. GUID_PERSONA
  10. SEXO
  11. NACIONALIDAD
  12. DEPARTAMENTO_NACIMIENTO
  13. ANIO_NACIMIENTO
  14. EDAD
  15. CODIGO_SIU_PROGRAMA
  16. CODIGO_GRUPO_1
  17. NOMBRE_GRUPO_1
  18. CODIGO_GRUPO_3
  19. NOMBRE_GRUPO_3
  20. NOMBRE_PROGRAMA
  21. ES_SEDE_PRINCIPAL
  22. CODIGO_SIU_FILIAL
  23. DEPARTAMENTO_FILIAL
  24. PROVINCIA_FILIAL
  25. CERT_GRAVEDAD
  26. DES_DISCAPACIDAD_DE_COMUNICACION
  27. DES_DISCAPACIDAD_DE_CONDUCTA
  28. DES_DISCAPACIDAD_DE_DESTREZA
  29. DES_DISCAPACIDAD_DE_DISPOSICION
  30. DES_DISCAPACIDAD_DE_LOCOMOCION
  31. DES_DISCAPACIDAD_DE_SITUACION
  32. DES_DISCAPACIDAD_DEL_CUIDADO

--- Comparación de dtypes ---
¿Mismos dtypes en los 6 archivos? -> True


In [10]:

def value_counts_por_archivo(reports, dir_path, col):
    filas = {}
    for name, r in reports.items():
        df = load_csv(dir_path / r["archivo"])
        filas[name] = df[col].value_counts(dropna=False).sort_index()
        del df
        gc.collect()
    return filas

print(f"--- Distribución de {period_col_ing} por archivo ---")
proc_counts = value_counts_por_archivo(ing_reportes, ING_DIR, period_col_ing)
display(pd.DataFrame(proc_counts).fillna(0).astype(int))


--- Distribución de PROCESO_ESTANDARIZADO por archivo ---


,ingresante_2020.csv,ingresante_2021.csv,ingresante_2022.csv,ingresante_2023.csv,ingresante_2024.csv,ingresante_2025.csv
PROCESO_ESTANDARIZADO,,,,,,
2020,434038,0,0,0,0,0
2021,0,483029,0,0,0,0
2022,0,0,490593,0,0,0
2023,0,0,0,536895,0,0
2024,0,0,0,0,565860,0
2025,0,0,0,0,0,609872


In [11]:

print("--- Nulos (todas las columnas) ---")
null_tab = pd.DataFrame({name: r["nulos"] for name, r in ing_reportes.items()}).T
rows = pd.Series({name: r["n_filas"] for name, r in ing_reportes.items()})
pct = (null_tab.div(rows, axis=0) * 100).round(2)
display(null_tab)
print("Porcentaje de nulos (%):")
display(pct)

print("Cardinalidad (valores únicos por columna):")
card_tab = pd.DataFrame({name: r["cardinalidad"] for name, r in ing_reportes.items()}).T
display(card_tab)


--- Nulos (todas las columnas) ---


,CODIGO_INEI,NOMBRE_ENTIDAD,TIPO_ENTIDAD,TIPO_GESTION,LICENCIADO,TIPO_CONSTITUCION,NIVEL_ACADEMICO,PROCESO_ESTANDARIZADO,GUID_PERSONA,SEXO,...,DEPARTAMENTO_FILIAL,PROVINCIA_FILIAL,CERT_GRAVEDAD,DES_DISCAPACIDAD_DE_COMUNICACION,DES_DISCAPACIDAD_DE_CONDUCTA,DES_DISCAPACIDAD_DE_DESTREZA,DES_DISCAPACIDAD_DE_DISPOSICION,DES_DISCAPACIDAD_DE_LOCOMOCION,DES_DISCAPACIDAD_DE_SITUACION,DES_DISCAPACIDAD_DEL_CUIDADO
ingresante_2020.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,25,432496,432496,432496,432496,432496,432496,432496
ingresante_2021.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,33,481167,481167,481167,481167,481167,481167,481167
ingresante_2022.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,22,488577,488577,488577,488577,488577,488577,488577
ingresante_2023.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,32,534583,534583,534583,534583,534583,534583,534583
ingresante_2024.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,40,563006,563006,563007,563007,563007,563007,563006
ingresante_2025.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,46,607075,607075,607075,607075,607075,607075,607075


Porcentaje de nulos (%):


,CODIGO_INEI,NOMBRE_ENTIDAD,TIPO_ENTIDAD,TIPO_GESTION,LICENCIADO,TIPO_CONSTITUCION,NIVEL_ACADEMICO,PROCESO_ESTANDARIZADO,GUID_PERSONA,SEXO,...,DEPARTAMENTO_FILIAL,PROVINCIA_FILIAL,CERT_GRAVEDAD,DES_DISCAPACIDAD_DE_COMUNICACION,DES_DISCAPACIDAD_DE_CONDUCTA,DES_DISCAPACIDAD_DE_DESTREZA,DES_DISCAPACIDAD_DE_DISPOSICION,DES_DISCAPACIDAD_DE_LOCOMOCION,DES_DISCAPACIDAD_DE_SITUACION,DES_DISCAPACIDAD_DEL_CUIDADO
ingresante_2020.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.64,99.64,99.64,99.64,99.64,99.64,99.64
ingresante_2021.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.61,99.61,99.61,99.61,99.61,99.61,99.61
ingresante_2022.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.59,99.59,99.59,99.59,99.59,99.59,99.59
ingresante_2023.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.57,99.57,99.57,99.57,99.57,99.57,99.57
ingresante_2024.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.50,99.50,99.50,99.50,99.50,99.50,99.50
ingresante_2025.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.54,99.54,99.54,99.54,99.54,99.54,99.54


Cardinalidad (valores únicos por columna):


,CODIGO_INEI,NOMBRE_ENTIDAD,TIPO_ENTIDAD,TIPO_GESTION,LICENCIADO,TIPO_CONSTITUCION,NIVEL_ACADEMICO,PROCESO_ESTANDARIZADO,GUID_PERSONA,SEXO,...,DEPARTAMENTO_FILIAL,PROVINCIA_FILIAL,CERT_GRAVEDAD,DES_DISCAPACIDAD_DE_COMUNICACION,DES_DISCAPACIDAD_DE_CONDUCTA,DES_DISCAPACIDAD_DE_DESTREZA,DES_DISCAPACIDAD_DE_DISPOSICION,DES_DISCAPACIDAD_DE_LOCOMOCION,DES_DISCAPACIDAD_DE_SITUACION,DES_DISCAPACIDAD_DEL_CUIDADO
ingresante_2020.csv,130,130,4,2,2,3,4,1,415073,2,...,25,79,5,8,7,8,8,8,8,8
ingresante_2021.csv,127,127,4,2,2,3,4,1,456720,2,...,25,74,5,8,7,8,8,8,8,8
ingresante_2022.csv,128,128,4,2,2,3,4,1,466592,2,...,25,81,5,7,8,8,8,8,8,8
ingresante_2023.csv,127,127,4,2,2,3,4,1,506748,2,...,25,82,5,8,8,8,7,8,8,8
ingresante_2024.csv,130,130,4,2,2,3,4,1,535605,2,...,25,80,5,8,8,8,8,8,8,8
ingresante_2025.csv,129,129,4,2,2,3,4,1,576594,2,...,25,79,5,8,8,8,8,8,8,8


In [12]:

print("--- Duplicados exactos (todas las columnas idénticas) ---")
display(ing_summary[["archivo", "n_filas", "duplicados_exactos"]])
print(f"Total duplicados exactos: {ing_summary['duplicados_exactos'].sum():,}")


--- Duplicados exactos (todas las columnas idénticas) ---


,archivo,n_filas,duplicados_exactos
0,ingresante_2020.csv,434038,0
1,ingresante_2021.csv,483029,0
2,ingresante_2022.csv,490593,0
3,ingresante_2023.csv,536895,0
4,ingresante_2024.csv,565860,0
5,ingresante_2025.csv,609872,0


Total duplicados exactos: 0


In [13]:

# Contrato para etl_pipeline.py: esquema dinámico de ingresantes
mejor_archivo = max(ing_reportes, key=lambda k: ing_reportes[k]["n_filas"])
clave_ing = ing_granularidad[mejor_archivo]["clave"]
dups_ing = ing_granularidad[mejor_archivo]["duplicados"]

ing_schema = build_schema(
    "ingresantes", ING_DIR, SEP, ENCODING, ing_reportes,
    clave_candidata=clave_ing, dups_clave=dups_ing,
    period_col=period_col_ing,
)
ruta_ing_schema = save_schema(ing_schema, SCHEMAS / "ingresantes_schema.json")
print("\nVista previa del contrato guardado:")
print(json.dumps(ing_schema, ensure_ascii=False, indent=2)[:1800])


Esquema guardado: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/schemas/ingresantes_schema.json

Vista previa del contrato guardado:
{
  "dataset": "ingresantes",
  "fuente": "/mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Bronce/ingresantes_raw",
  "n_archivos": 6,
  "n_filas_total": 3120287,
  "encoding": "latin-1",
  "separador": "|",
  "columna_periodo": "PROCESO_ESTANDARIZADO",
  "columnas": [
    "CODIGO_INEI",
    "NOMBRE_ENTIDAD",
    "TIPO_ENTIDAD",
    "TIPO_GESTION",
    "LICENCIADO",
    "TIPO_CONSTITUCION",
    "NIVEL_ACADEMICO",
    "PROCESO_ESTANDARIZADO",
    "GUID_PERSONA",
    "SEXO",
    "NACIONALIDAD",
    "DEPARTAMENTO_NACIMIENTO",
    "ANIO_NACIMIENTO",
    "EDAD",
    "CODIGO_SIU_PROGRAMA",
    "CODIGO_GRUPO_1",
    "NOMBRE_GRUPO_1",
    "CODIGO_GRUPO_3",
    "NOMBRE_GRUPO_3",
    "NOMBRE_PROGRAMA",
    "ES_SEDE_PRINCIPAL",
    "CODIGO_SIU_FILIAL",
    "DEPARTAMENTO_FILIAL",
    "PROVINCIA_FILIAL",
    "CERT_GRAVEDAD",
    "DES_DISCAPACIDAD_DE_COMUNICACION",
 

In [14]:

same_cols = all(r["columnas"] == list(ing_reportes.values())[0]["columnas"]
                for r in ing_reportes.values())
same_dtypes = all(r["dtypes"] == list(ing_reportes.values())[0]["dtypes"]
                  for r in ing_reportes.values())
print("ESQUEMA COMPATIBLE para futura concatenación")
print(f"  ¿Columnas idénticas en los {len(ing_reportes)} archivos? -> {same_cols}")
print(f"  ¿Dtypes idénticos en los {len(ing_reportes)} archivos?  -> {same_dtypes}")
print(f"RESULTADO CONCATENABLE: {'SÍ' if (same_cols and same_dtypes) else 'NO'}")
print("NOTA: no se concatena en este notebook; se difiere a etl_pipeline.py.")
print(f"Clave candidata detectada (hipótesis): {' + '.join(clave_ing)} "
      f"| duplicados: {dups_ing:,}")


ESQUEMA COMPATIBLE para futura concatenación
  ¿Columnas idénticas en los 6 archivos? -> True
  ¿Dtypes idénticos en los 6 archivos?  -> True
RESULTADO CONCATENABLE: SÍ
NOTA: no se concatena en este notebook; se difiere a etl_pipeline.py.
Clave candidata detectada (hipótesis): CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA | duplicados: 0



# PARTE 2 — MATRICULADOS

Mismo análisis dinámico para los archivos de matriculados (periodos I y II por año). Carga secuencial, uno a la vez.


In [15]:

MAT_FILES = sorted(MAT_DIR.glob("*.csv"))
print(f"Archivos encontrados: {len(MAT_FILES)}")
for f in MAT_FILES:
    print(f"  {f.name:28s} {f.stat().st_size/1e6:9.1f} MB")


Archivos encontrados: 12
  matriculado_2020_I.csv           491.3 MB
  matriculado_2020_II.csv          474.0 MB
  matriculado_2021_I.csv           546.1 MB
  matriculado_2021_II.csv          545.6 MB
  matriculado_2022_I.csv           585.4 MB
  matriculado_2022_II.csv          557.9 MB
  matriculado_2023_I.csv           590.0 MB
  matriculado_2023_II.csv          559.0 MB
  matriculado_2024_I.csv           617.2 MB
  matriculado_2024_II.csv          604.3 MB
  matriculado_2025_I.csv           659.0 MB
  matriculado_2025_II.csv          636.0 MB


In [16]:

mat_reportes = {}
mat_granularidad = {}
period_col_mat = None

for f in MAT_FILES:
    df, info = inspect_csv(f)
    if period_col_mat is None:
        period_col_mat = detect_period_col(df)
    cand = detect_identifier_candidates(df)
    clave, dups = find_greedy_key(df, cand)
    mat_granularidad[f.name] = {
        "candidatos": cand, "clave": clave, "duplicados": dups,
    }
    mat_reportes[f.name] = info
    del df
    gc.collect()

print("Inspección completada. Reportes en 'mat_reportes'.")
print(f"Columna de periodo detectada: {period_col_mat}")


Inspección completada. Reportes en 'mat_reportes'.
Columna de periodo detectada: PERIODO_ESTANDARIZADO


In [17]:

mat_summary = tabla_resumen(mat_reportes)
display(mat_summary)
print(f"Total filas matriculados: {mat_summary['n_filas'].sum():,}")

print("\nGranularidad detectada por archivo (hipótesis por cardinalidad):")
g = pd.DataFrame(mat_granularidad).T
g["candidatos"] = g["candidatos"].apply(lambda x: ", ".join(x))
g["clave"] = g["clave"].apply(lambda x: " + ".join(x))
display(g[["candidatos", "clave", "duplicados"]])


,archivo,n_filas,n_columnas,duplicados_exactos
0,matriculado_2020_I.csv,1304115,39,78649
1,matriculado_2020_II.csv,1262299,39,76524
2,matriculado_2021_I.csv,1454409,39,75374
3,matriculado_2021_II.csv,1453084,39,74965
4,matriculado_2022_I.csv,1559519,39,77934
5,matriculado_2022_II.csv,1485856,39,74349
6,matriculado_2023_I.csv,1570240,39,76361
7,matriculado_2023_II.csv,1486902,39,71918
8,matriculado_2024_I.csv,1641259,39,74385
9,matriculado_2024_II.csv,1607876,39,72449


Total filas matriculados: 18,270,871

Granularidad detectada por archivo (hipótesis por cardinalidad):


,candidatos,clave,duplicados
matriculado_2020_I.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,78649
matriculado_2020_II.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,76524
matriculado_2021_I.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,75374
matriculado_2021_II.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,74965
matriculado_2022_I.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,77934
matriculado_2022_II.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,74349
matriculado_2023_I.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,76361
matriculado_2023_II.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,71918
matriculado_2024_I.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,74385
matriculado_2024_II.csv,"CODIGO_INEI, CODIGO_SIU_PROGRAMA, CODIGO_GRUPO...",CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRU...,72449


In [18]:

base = list(mat_reportes.values())[0]
print("--- Comparación de columnas ---")
same_cols = all(r["columnas"] == base["columnas"] for r in mat_reportes.values())
print(f"¿Mismas columnas en los {len(mat_reportes)} archivos? -> {same_cols}")
if same_cols:
    print(f"Columnas ({len(base['columnas'])}):")
    for i, c in enumerate(base["columnas"], 1):
        print(f"  {i:2d}. {c}")

print("\n--- Comparación de dtypes ---")
same_dtypes = all(r["dtypes"] == base["dtypes"] for r in mat_reportes.values())
print(f"¿Mismos dtypes en los {len(mat_reportes)} archivos? -> {same_dtypes}")
if not same_dtypes:
    diffs = {}
    for name, r in mat_reportes.items():
        for c, dt in r["dtypes"].items():
            diffs.setdefault(c, set()).add(dt)
    print("Columnas con dtype inconsistente:",
          {c: sorted(v) for c, v in diffs.items() if len(v) > 1})


--- Comparación de columnas ---
¿Mismas columnas en los 12 archivos? -> True
Columnas (39):
   1. CODIGO_INEI
   2. NOMBRE_ENTIDAD
   3. TIPO_ENTIDAD
   4. TIPO_GESTION
   5. TIPO_CONSTITUCION
   6. LICENCIA
   7. PERIODO
   8. PERIODO_ESTANDARIZADO
   9. NIVEL_ACADEMICO
  10. PERIODO_LECTIVO
  11. CODIGO_SIU_PROGRAMA
  12. CODIGO_GRUPO_1
  13. NOMBRE_GRUPO_1
  14. CODIGO_GRUPO_3
  15. NOMBRE_GRUPO_3
  16. NOMBRE_PROGRAMA
  17. ES_LOCAL_PRINCIPAL
  18. CODIGO_LOCAL
  19. DEPARTAMENTO_LOCAL
  20. PROVINCIA_LOCAL
  21. DISTRITO_LOCAL
  22. GUID_PERSONA
  23. SEXO
  24. ANIO_NACIMIENTO
  25. EDAD
  26. NACIONALIDAD
  27. DEPARTAMENTO_NACIMIENTO
  28. ANIO_PERIODO_INGRESO
  29. FECHA_INICIO_PERIODO
  30. FECHA_FIN_PERIODO
  31. CODIGO_UBIGEO_INEI_LOCAL
  32. CERT_GRAVEDAD
  33. DES_DISCAPACIDAD_DE_COMUNICACION
  34. DES_DISCAPACIDAD_DE_CONDUCTA
  35. DES_DISCAPACIDAD_DE_DESTREZA
  36. DES_DISCAPACIDAD_DE_DISPOSICION
  37. DES_DISCAPACIDAD_DE_LOCOMOCION
  38. DES_DISCAPACIDAD_DE_SITUACION
 

In [19]:

print(f"--- Distribución de {period_col_mat} por archivo ---")
peri_counts = value_counts_por_archivo(mat_reportes, MAT_DIR, period_col_mat)
display(pd.DataFrame(peri_counts).fillna(0).astype(int))


--- Distribución de PERIODO_ESTANDARIZADO por archivo ---


,matriculado_2020_I.csv,matriculado_2020_II.csv,matriculado_2021_I.csv,matriculado_2021_II.csv,matriculado_2022_I.csv,matriculado_2022_II.csv,matriculado_2023_I.csv,matriculado_2023_II.csv,matriculado_2024_I.csv,matriculado_2024_II.csv,matriculado_2025_I.csv,matriculado_2025_II.csv
PERIODO_ESTANDARIZADO,,,,,,,,,,,,
2020-1,1304115,0,0,0,0,0,0,0,0,0,0,0
2020-2,0,1262299,0,0,0,0,0,0,0,0,0,0
2021-1,0,0,1454409,0,0,0,0,0,0,0,0,0
2021-2,0,0,0,1453084,0,0,0,0,0,0,0,0
2022-1,0,0,0,0,1559519,0,0,0,0,0,0,0
2022-2,0,0,0,0,0,1485856,0,0,0,0,0,0
2023-1,0,0,0,0,0,0,1570240,0,0,0,0,0
2023-2,0,0,0,0,0,0,0,1486902,0,0,0,0
2024-1,0,0,0,0,0,0,0,0,1641259,0,0,0


In [20]:

print("--- Nulos (todas las columnas) ---")
null_tab = pd.DataFrame({name: r["nulos"] for name, r in mat_reportes.items()}).T
rows = pd.Series({name: r["n_filas"] for name, r in mat_reportes.items()})
pct = (null_tab.div(rows, axis=0) * 100).round(2)
display(null_tab)
print("Porcentaje de nulos (%):")
display(pct)

print("Cardinalidad (valores únicos por columna):")
card_tab = pd.DataFrame({name: r["cardinalidad"] for name, r in mat_reportes.items()}).T
display(card_tab)


--- Nulos (todas las columnas) ---


,CODIGO_INEI,NOMBRE_ENTIDAD,TIPO_ENTIDAD,TIPO_GESTION,TIPO_CONSTITUCION,LICENCIA,PERIODO,PERIODO_ESTANDARIZADO,NIVEL_ACADEMICO,PERIODO_LECTIVO,...,FECHA_FIN_PERIODO,CODIGO_UBIGEO_INEI_LOCAL,CERT_GRAVEDAD,DES_DISCAPACIDAD_DE_COMUNICACION,DES_DISCAPACIDAD_DE_CONDUCTA,DES_DISCAPACIDAD_DE_DESTREZA,DES_DISCAPACIDAD_DE_DISPOSICION,DES_DISCAPACIDAD_DE_LOCOMOCION,DES_DISCAPACIDAD_DE_SITUACION,DES_DISCAPACIDAD_DEL_CUIDADO
matriculado_2020_I.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,60,1300332,1300332,1300332,1300332,1300332,1300332,1300332
matriculado_2020_II.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,68,1258526,1258526,1258526,1258526,1258526,1258526,1258526
matriculado_2021_I.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,71,1449870,1449870,1449870,1449870,1449870,1449870,1449870
matriculado_2021_II.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,67,1448549,1448549,1448549,1448549,1448549,1448550,1448549
matriculado_2022_I.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,66,1554444,1554444,1554444,1554444,1554444,1554444,1554444
matriculado_2022_II.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,71,1481016,1481016,1481016,1481016,1481016,1481016,1481016
matriculado_2023_I.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,70,1564842,1564842,1564842,1564842,1564842,1564842,1564842
matriculado_2023_II.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,74,1481708,1481708,1481708,1481708,1481708,1481708,1481708
matriculado_2024_I.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,85,1634880,1634880,1634881,1634881,1634881,1634881,1634880
matriculado_2024_II.csv,0,0,0,0,0,0,0,0,0,0,...,0,0,87,1601466,1601466,1601467,1601467,1601467,1601467,1601466


Porcentaje de nulos (%):


,CODIGO_INEI,NOMBRE_ENTIDAD,TIPO_ENTIDAD,TIPO_GESTION,TIPO_CONSTITUCION,LICENCIA,PERIODO,PERIODO_ESTANDARIZADO,NIVEL_ACADEMICO,PERIODO_LECTIVO,...,FECHA_FIN_PERIODO,CODIGO_UBIGEO_INEI_LOCAL,CERT_GRAVEDAD,DES_DISCAPACIDAD_DE_COMUNICACION,DES_DISCAPACIDAD_DE_CONDUCTA,DES_DISCAPACIDAD_DE_DESTREZA,DES_DISCAPACIDAD_DE_DISPOSICION,DES_DISCAPACIDAD_DE_LOCOMOCION,DES_DISCAPACIDAD_DE_SITUACION,DES_DISCAPACIDAD_DEL_CUIDADO
matriculado_2020_I.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.71,99.71,99.71,99.71,99.71,99.71,99.71
matriculado_2020_II.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.70,99.70,99.70,99.70,99.70,99.70,99.70
matriculado_2021_I.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.69,99.69,99.69,99.69,99.69,99.69,99.69
matriculado_2021_II.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.69,99.69,99.69,99.69,99.69,99.69,99.69
matriculado_2022_I.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.67,99.67,99.67,99.67,99.67,99.67,99.67
matriculado_2022_II.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.67,99.67,99.67,99.67,99.67,99.67,99.67
matriculado_2023_I.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.66,99.66,99.66,99.66,99.66,99.66,99.66
matriculado_2023_II.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,99.65,99.65,99.65,99.65,99.65,99.65,99.65
matriculado_2024_I.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.61,99.61,99.61,99.61,99.61,99.61,99.61
matriculado_2024_II.csv,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.01,99.60,99.60,99.60,99.60,99.60,99.60,99.60


Cardinalidad (valores únicos por columna):


,CODIGO_INEI,NOMBRE_ENTIDAD,TIPO_ENTIDAD,TIPO_GESTION,TIPO_CONSTITUCION,LICENCIA,PERIODO,PERIODO_ESTANDARIZADO,NIVEL_ACADEMICO,PERIODO_LECTIVO,...,FECHA_FIN_PERIODO,CODIGO_UBIGEO_INEI_LOCAL,CERT_GRAVEDAD,DES_DISCAPACIDAD_DE_COMUNICACION,DES_DISCAPACIDAD_DE_CONDUCTA,DES_DISCAPACIDAD_DE_DESTREZA,DES_DISCAPACIDAD_DE_DISPOSICION,DES_DISCAPACIDAD_DE_LOCOMOCION,DES_DISCAPACIDAD_DE_SITUACION,DES_DISCAPACIDAD_DEL_CUIDADO
matriculado_2020_I.csv,168,168,4,2,3,3,2,1,4,5,...,138,177,5,8,8,8,8,8,8,8
matriculado_2020_II.csv,166,166,4,2,3,3,3,1,4,5,...,127,168,5,8,8,8,8,8,8,8
matriculado_2021_I.csv,165,165,4,2,3,3,2,1,4,5,...,141,168,5,8,8,8,8,8,8,8
matriculado_2021_II.csv,159,159,4,2,3,3,3,1,4,5,...,141,167,5,8,8,8,8,8,8,8
matriculado_2022_I.csv,149,149,4,2,3,3,2,1,4,5,...,142,160,5,8,8,8,8,8,8,8
matriculado_2022_II.csv,147,147,4,2,3,3,4,1,4,6,...,141,157,5,8,8,8,8,8,8,8
matriculado_2023_I.csv,144,144,4,2,3,3,2,1,4,6,...,124,157,5,8,8,8,8,8,8,8
matriculado_2023_II.csv,141,141,4,2,3,3,4,1,4,6,...,120,154,5,8,8,8,8,8,8,8
matriculado_2024_I.csv,140,140,4,2,3,3,2,1,4,6,...,139,158,5,8,8,8,8,8,8,8
matriculado_2024_II.csv,139,139,4,2,3,3,4,1,4,6,...,128,159,5,8,8,8,8,8,8,8


In [21]:

print("--- Duplicados exactos (todas las columnas idénticas) ---")
display(mat_summary[["archivo", "n_filas", "duplicados_exactos"]])
print(f"Total duplicados exactos: {mat_summary['duplicados_exactos'].sum():,}")


--- Duplicados exactos (todas las columnas idénticas) ---


,archivo,n_filas,duplicados_exactos
0,matriculado_2020_I.csv,1304115,78649
1,matriculado_2020_II.csv,1262299,76524
2,matriculado_2021_I.csv,1454409,75374
3,matriculado_2021_II.csv,1453084,74965
4,matriculado_2022_I.csv,1559519,77934
5,matriculado_2022_II.csv,1485856,74349
6,matriculado_2023_I.csv,1570240,76361
7,matriculado_2023_II.csv,1486902,71918
8,matriculado_2024_I.csv,1641259,74385
9,matriculado_2024_II.csv,1607876,72449


Total duplicados exactos: 902,447


In [22]:

# Contrato para etl_pipeline.py: esquema dinámico de matriculados
mejor_archivo = max(mat_reportes, key=lambda k: mat_reportes[k]["n_filas"])
clave_mat = mat_granularidad[mejor_archivo]["clave"]
dups_mat = mat_granularidad[mejor_archivo]["duplicados"]

mat_schema = build_schema(
    "matriculados", MAT_DIR, SEP, ENCODING, mat_reportes,
    clave_candidata=clave_mat, dups_clave=dups_mat,
    period_col=period_col_mat,
)
ruta_mat_schema = save_schema(mat_schema, SCHEMAS / "matriculados_schema.json")
print("\nVista previa del contrato guardado:")
print(json.dumps(mat_schema, ensure_ascii=False, indent=2)[:1800])


Esquema guardado: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/schemas/matriculados_schema.json

Vista previa del contrato guardado:
{
  "dataset": "matriculados",
  "fuente": "/mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Bronce/matriculados_raw",
  "n_archivos": 12,
  "n_filas_total": 18270871,
  "encoding": "latin-1",
  "separador": "|",
  "columna_periodo": "PERIODO_ESTANDARIZADO",
  "columnas": [
    "CODIGO_INEI",
    "NOMBRE_ENTIDAD",
    "TIPO_ENTIDAD",
    "TIPO_GESTION",
    "TIPO_CONSTITUCION",
    "LICENCIA",
    "PERIODO",
    "PERIODO_ESTANDARIZADO",
    "NIVEL_ACADEMICO",
    "PERIODO_LECTIVO",
    "CODIGO_SIU_PROGRAMA",
    "CODIGO_GRUPO_1",
    "NOMBRE_GRUPO_1",
    "CODIGO_GRUPO_3",
    "NOMBRE_GRUPO_3",
    "NOMBRE_PROGRAMA",
    "ES_LOCAL_PRINCIPAL",
    "CODIGO_LOCAL",
    "DEPARTAMENTO_LOCAL",
    "PROVINCIA_LOCAL",
    "DISTRITO_LOCAL",
    "GUID_PERSONA",
    "SEXO",
    "ANIO_NACIMIENTO",
    "EDAD",
    "NACIONALIDAD",
    "DEPARTAMENTO_NACIMIENTO",
    "

In [23]:

same_cols = all(r["columnas"] == list(mat_reportes.values())[0]["columnas"]
                for r in mat_reportes.values())
same_dtypes = all(r["dtypes"] == list(mat_reportes.values())[0]["dtypes"]
                  for r in mat_reportes.values())
print("ESQUEMA COMPATIBLE para futura concatenación")
print(f"  ¿Columnas idénticas en los {len(mat_reportes)} archivos? -> {same_cols}")
print(f"  ¿Dtypes idénticos en los {len(mat_reportes)} archivos?  -> {same_dtypes}")
print(f"RESULTADO CONCATENABLE: {'SÍ' if (same_cols and same_dtypes) else 'NO'}")
print("NOTA: no se concatena en este notebook; se difiere a etl_pipeline.py.")
print(f"Clave candidata detectada (hipótesis): {' + '.join(clave_mat)} "
      f"| duplicados: {dups_mat:,}")


ESQUEMA COMPATIBLE para futura concatenación
  ¿Columnas idénticas en los 12 archivos? -> True
  ¿Dtypes idénticos en los 12 archivos?  -> True
RESULTADO CONCATENABLE: SÍ
NOTA: no se concatena en este notebook; se difiere a etl_pipeline.py.
Clave candidata detectada (hipótesis): CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRUPO_1 + CODIGO_GRUPO_3 + CODIGO_LOCAL + GUID_PERSONA | duplicados: 75,907



# PARTE 3 — institucional.csv (Oferta Universitaria)

Documentación de su estructura real y del problema de parsing por **comas internas no escapadas**. No se asume que ningún parser heurístico sea correcto al 100%: se documentan porcentajes y ambigüedades.


In [24]:

with INST_FILE.open("r", encoding=INST_ENCODING) as fh:
    header = fh.readline().rstrip("\n")
    lineas = [l.rstrip("\n") for l in fh]

inst_cols = header.split(",")
print(f"Filas de datos: {len(lineas):,} | Columnas declaradas en la cabecera: {len(inst_cols)}")
for i, c in enumerate(inst_cols, 1):
    print(f"  {i:2d}. {c}")


Filas de datos: 12,446 | Columnas declaradas en la cabecera: 27
   1. ENTIDAD_CODIGO_INEI
   2. ENTIDAD_NOMBRE_ENTIDAD
   3. ENTIDAD_TIPO_ENTIDAD
   4. ENTIDAD_TIPO_AUTORIZACION
   5. ENTIDAD_TIPO_CONSTITUCION
   6. ENTIDAD_TIPO_GESTION
   7. ENTIDAD_FECHA_RESOLUCION_ENTIDAD
   8. ENTIDAD_DURACION_LICENCIAMIENTO
   9. PROGRAMA_CODIGO
  10. PROGRAMA_CODIGO_GRUPO_1
  11. PROGRAMA_NOMBRE_GRUPO_1
  12. PROGRAMA_CODIGO_GRUPO_3
  13. PROGRAMA_NOMBRE_GRUPO_3
  14. PROGRAMA_NOMBRE
  15. PROGRAMA_NIVEL_ACADEMICO
  16. PROGRAMA_DURACION
  17. PROGRAMA_TIPO_AUTORIZACION_LOCAL
  18. LOCAL_CODIGO
  19. LOCAL_DEPARTAMENTO
  20. LOCAL_PROVINCIA
  21. LOCAL_DISTRITO
  22. LOCAL_DIRECCION
  23. LOCAL_ES_PRINCIPAL
  24. LOCAL_LONGITUD_UBICACION
  25. LOCAL_LATITUD_UBICACION
  26. LOCAL_CODIGO_UBIGEO_INEI_LOCAL
  27. LOCAL_MODALIDAD_ESTUDIO


In [25]:

conteo = Counter(l.count(",") + 1 for l in lineas)
print("Distribución de campos por línea (la cabecera declara 27):")
for n in sorted(conteo):
    marca = "   <<< coincide con cabecera" if n == len(inst_cols) else ""
    print(f"  {n} campos: {conteo[n]:>6,} líneas{marca}")
print(f"\nLíneas que respetan los {len(inst_cols)} campos: "
      f"{conteo.get(len(inst_cols), 0):,} de {len(lineas):,}")
print("=> El parseo estándar (pd.read_csv) NO es viable tal cual.")


Distribución de campos por línea (la cabecera declara 27):
  27 campos:      1 líneas   <<< coincide con cabecera
  28 campos:      3 líneas
  29 campos:  1,270 líneas
  30 campos:  5,181 líneas
  31 campos:  3,964 líneas
  32 campos:  1,329 líneas
  33 campos:    394 líneas
  34 campos:    161 líneas
  35 campos:     33 líneas
  36 campos:      2 líneas
  39 campos:     20 líneas
  40 campos:     68 líneas
  41 campos:     18 líneas
  42 campos:      2 líneas

Líneas que respetan los 27 campos: 1 de 12,446
=> El parseo estándar (pd.read_csv) NO es viable tal cual.


In [26]:

print("Evidencia: las comas extra provienen de columnas TEXTUALES sin comillas de escape.")
print("Fuentes del problema (identificadas por inspección de las líneas):")
print("  1) PROGRAMA_NOMBRE_GRUPO_1  -> 'Ciencias Sociales, Periodismo e Información'")
print("  2) PROGRAMA_NOMBRE_GRUPO_3  -> 'Hotelería, Restaurantes y Gastronomía'")
print("  3) PROGRAMA_NOMBRE          -> 'Educación, Especialidad Educación Física, Recreación y Deportes'")
print("  4) LOCAL_DIRECCION          -> direcciones con listados ('LOTES N° 27, 27-A, 28, ...')")
print()
mostradas = set()
for l in lineas:
    n = l.count(",") + 1
    if n in (27, 30, 32, 42) and n not in mostradas:
        mostradas.add(n)
        print(f"--- Línea real con {n} campos ---")
        print(l[:220])
        print()


Evidencia: las comas extra provienen de columnas TEXTUALES sin comillas de escape.
Fuentes del problema (identificadas por inspección de las líneas):
  1) PROGRAMA_NOMBRE_GRUPO_1  -> 'Ciencias Sociales, Periodismo e Información'
  2) PROGRAMA_NOMBRE_GRUPO_3  -> 'Hotelería, Restaurantes y Gastronomía'
  3) PROGRAMA_NOMBRE          -> 'Educación, Especialidad Educación Física, Recreación y Deportes'
  4) LOCAL_DIRECCION          -> direcciones con listados ('LOTES N° 27, 27-A, 28, ...')

--- Línea real con 30 campos ---
160000001,Universidad Nacional Mayor de San Marcos,Universidad,Licenciada,Pública,Público,12/05/1551,10,1,4,Ciencias Administrativas y Derecho,413,Gestión y Administración,Administración,Carrera Profesional,5,00,Declarad

--- Línea real con 32 campos ---
160000001,Universidad Nacional Mayor de San Marcos,Universidad,Licenciada,Pública,Público,12/05/1551,10,48,8,Agricultura, Silvicultura, Pesca y Veterinaria,841,Veterinaria,Ciencias Veterinarias,Maestría,2,00,Reconocido p


In [27]:

# Parser estructural por 'anclas' (SOLUCIÓN PARCIAL y documentada como tal).
NIVEL = {"Carrera Profesional", "Segunda Especialidad", "Maestría", "Doctorado"}


def is_int(t):
    return re.fullmatch(r"-?\d+", t) is not None


def parse_anclas(line):
    """Reconstruye 27 campos por anclas. Devuelve (row, frontera_13_14_confiable).
    NO garantiza la asignación correcta de las columnas 13/14 cuando ambas
    columnas de nombre llevan coma interna (ambigüedad estructural)."""
    t = line.split(",")
    i = 0
    out = [None] * 27
    for k in range(10):  # cols 1..10 fijas
        if i >= len(t):
            return None, False
        out[k] = t[i]; i += 1
    acc = [t[i]]; i += 1  # col 11 (NOMBRE_GRUPO_1) hasta el entero de col 12
    while i < len(t) and not is_int(t[i]):
        acc.append(t[i]); i += 1
    out[10] = " ".join(acc)
    if i >= len(t):
        return None, False
    out[11] = t[i]; i += 1  # col 12 (CODIGO_GRUPO_3) fija
    acc = [t[i]]; i += 1    # cols 13-14 variables hasta el nivel académico (col 15)
    while i < len(t) and t[i] not in NIVEL:
        acc.append(t[i]); i += 1
    frontera_confiable = len(acc) == 1
    out[12] = acc[0]
    out[13] = " ".join(acc[1:])
    if i >= len(t):
        return None, False
    out[14] = t[i]; i += 1  # col 15 (NIVEL_ACADEMICO)
    for k in range(15, 22):  # cols 16..21 fijas
        if i >= len(t):
            return None, False
        out[k] = t[i]; i += 1
    acc = [t[i]]; i += 1    # col 22 (LOCAL_DIRECCION) hasta el token SI/NO (col 23)
    while i < len(t) and t[i] not in ("SI", "NO"):
        acc.append(t[i]); i += 1
    out[21] = " ".join(acc)
    for k in range(22, 27):  # cols 23..27 fijas
        if i >= len(t):
            return None, False
        out[k] = t[i]; i += 1
    if i != len(t):
        return None, False
    return out, frontera_confiable


res = [parse_anclas(l) for l in lineas]
recon_ok = sum(1 for r, _ in res if r is not None)
frontera_ok = sum(1 for r, b in res if r is not None and b)
print(f"Total de líneas: {len(lineas):,}")
print(f"  Estructura reconstruida (27 campos, sin desborde): {recon_ok:,} "
      f"({recon_ok/len(lineas)*100:.1f}%)")
print(f"    -> de las cuales, frontera col13/col14 CONFIABLE: {frontera_ok:,} "
      f"({frontera_ok/len(lineas)*100:.1f}%)")
print(f"  Fallas (estructura no reconstruible): {len(lineas)-recon_ok:,} "
      f"({(len(lineas)-recon_ok)/len(lineas)*100:.1f}%)")
print("\nCONCLUSIÓN: el parser por anclas es una SOLUCIÓN PARCIAL. La frontera")
print("col13/col14 es ambigua y NO puede resolverse sin referencia externa.")


Total de líneas: 12,446
  Estructura reconstruida (27 campos, sin desborde): 2 (0.0%)
    -> de las cuales, frontera col13/col14 CONFIABLE: 0 (0.0%)
  Fallas (estructura no reconstruible): 12,444 (100.0%)

CONCLUSIÓN: el parser por anclas es una SOLUCIÓN PARCIAL. La frontera
col13/col14 es ambigua y NO puede resolverse sin referencia externa.


In [28]:

print("Ejemplo concreto de ambigüedad col13 / col14:")
for l in lineas:
    if "764" in l and "Hotelería" in l and "Administración de la Gastronomía" in l:
        t = l.split(",")
        print("  Línea real:", l[:180], "...")
        print()
        print(f"  col 12 (CODIGO_GRUPO_3)  = {t[11]}")
        print(f"  col 13 (NOMBRE_GRUPO_3)  = 'Hotelería, Restaurantes y Gastronomía'  <- coma interna")
        print(f"  col 14 (PROGRAMA_NOMBRE) = 'Administración de la Gastronomía'")
        print(f"  col 15 (NIVEL_ACADEMICO) = 'Carrera Profesional'")
        print()
        print("  El parser no puede decidir dónde termina col 13 y empieza col 14 sin el")
        print("  catálogo INEI-2022 (codigo_grupo_3 -> nombre canónico exacto).")
        break


Ejemplo concreto de ambigüedad col13 / col14:
  Línea real: 160000001,Universidad Nacional Mayor de San Marcos,Universidad,Licenciada,Pública,Público,12/05/1551,10,764,0,Servicios,012,Hotelería, Restaurantes y Gastronomía,Administración de  ...

  col 12 (CODIGO_GRUPO_3)  = 012
  col 13 (NOMBRE_GRUPO_3)  = 'Hotelería, Restaurantes y Gastronomía'  <- coma interna
  col 14 (PROGRAMA_NOMBRE) = 'Administración de la Gastronomía'
  col 15 (NIVEL_ACADEMICO) = 'Carrera Profesional'

  El parser no puede decidir dónde termina col 13 y empieza col 14 sin el
  catálogo INEI-2022 (codigo_grupo_3 -> nombre canónico exacto).


In [29]:

print("""
CONCLUSIÓN — institucional.csv
- Es el catálogo de Oferta Universitaria (sección 5.9 de la ficha TUNI): entidad, programa y local.
- Problema de calidad: comas sin escapar en PROGRAMA_NOMBRE_GRUPO_1/3, PROGRAMA_NOMBRE y LOCAL_DIRECCION.
- El parser por anclas reconstruye la estructura de la mayoría de líneas, pero la frontera
  col 13/col 14 es AMBIGUA cuando NOMBRE_GRUPO_3 lleva coma interna.
- Siguiente paso (NO dependencia del pipeline actual): usar el catálogo INEI-2022 para mapear
  CODIGO_GRUPO_1 / CODIGO_GRUPO_3 -> nombre canónico exacto y eliminar la ambigüedad.
- Uso recomendado: OPCIONAL. No bloquea las preguntas obligatorias.
""")



CONCLUSIÓN — institucional.csv
- Es el catálogo de Oferta Universitaria (sección 5.9 de la ficha TUNI): entidad, programa y local.
- Problema de calidad: comas sin escapar en PROGRAMA_NOMBRE_GRUPO_1/3, PROGRAMA_NOMBRE y LOCAL_DIRECCION.
- El parser por anclas reconstruye la estructura de la mayoría de líneas, pero la frontera
  col 13/col 14 es AMBIGUA cuando NOMBRE_GRUPO_3 lleva coma interna.
- Siguiente paso (NO dependencia del pipeline actual): usar el catálogo INEI-2022 para mapear
  CODIGO_GRUPO_1 / CODIGO_GRUPO_3 -> nombre canónico exacto y eliminar la ambigüedad.
- Uso recomendado: OPCIONAL. No bloquea las preguntas obligatorias.



# Conclusiones y decisiones (RESERVADO)

## 1. Qué fuentes podemos usar

**Ingresantes — ✅ Concatenable, alta calidad.**
Los 6 archivos (2020-2025) tienen columnas y dtypes idénticos. 0 duplicados exactos en todo el dataset. La clave candidata `CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA` logra 0 duplicados — es una clave robusta y coherente con el significado del dataset ("un ingresante se matricula una vez por proceso de admisión a un programa"). Apto para concatenar sin transformación previa más allá de tipado y limpieza estándar.

**Matriculados — ⚠️ Concatenable, pero requiere deduplicación explícita antes de usarse.**
Los 12 archivos son estructuralmente consistentes (mismas 39 columnas/dtypes). Sin embargo:
- 902,447 duplicados exactos (~4.94% del total) — se resuelven con un simple `drop_duplicates()`.
- 75,907 duplicados bajo la clave candidata **incluso después de** deduplicar exactos — esto indica que la clave propuesta aún no captura la granularidad real del dataset (ver punto 3, es un hallazgo importante).

## 2. Fuentes que requieren tratamiento especial

- **institucional.csv — descartada del pipeline principal.** Solo 1 de 12,446 líneas respeta el número de columnas de la cabecera; el parser heurístico por anclas solo reconstruye 2 líneas (0.02%). La causa raíz (comas internas sin escapar en campos como nombres de grupo — ej. "Hotelería, Restaurantes y Gastronomía") genera ambigüedad estructural real, no solo un problema de parseo. **No bloquea las preguntas obligatorias**, así que se documenta como limitación y se deja como *siguiente paso* condicionado a conseguir el catálogo INEI-2022 para desambiguar por matching de nombres.
- **Matriculados** requiere tratamiento de deduplicación en dos capas (exactos + clave candidata) antes de poder considerarse "limpio" para el modelo dimensional.

## 3. Columnas clave detectadas (granularidad)

| Dataset | Clave candidata | Resultado | Confianza |
|---|---|---|---|
| Ingresantes | `CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA` | 0 duplicados | Alta, pero pendiente de validación de negocio |
| Matriculados | `CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRUPO_1 + CODIGO_GRUPO_3 + CODIGO_LOCAL + GUID_PERSONA` | 75,907 duplicados | **Baja — clave incompleta** |

**Hallazgo importante a documentar explícitamente:** la clave candidata de `matriculados` **no incluye `PERIODO_ESTANDARIZADO`**, pese a que el dataset consolida 12 archivos que cubren 6 años × 2 semestres. Esto es una señal de alerta, no un detalle menor:

- Si el algoritmo voraz corrió sobre el dataset ya concatenado (multi-periodo) y aun así no incorporó el periodo a la clave, hay dos lecturas posibles: (a) el mismo estudiante rara vez repite exactamente la misma combinación de programa/grupo/local en dos periodos distintos, por lo que la clave "funciona" casi por casualidad, no por diseño; o (b) el algoritmo se detuvo en un óptimo local antes de evaluar `PERIODO_ESTANDARIZADO` como columna candidata (por ejemplo, si su cardinalidad relativa quedó por debajo del umbral de 30% usado para preselección).
- Para un dashboard de **series por periodo** (que es justamente el objetivo del caso), la clave de negocio *correcta* casi con certeza debe incluir el periodo — de lo contrario, dos registros del mismo estudiante en el mismo programa/grupo/local pero en semestres distintos podrían tratarse como el mismo evento de matrícula, o el proceso de deduplicación por clave podría eliminar por error registros de periodos distintos que sí son válidos.
- **Decisión recomendada:** forzar `PERIODO_ESTANDARIZADO` dentro de la clave de matriculados para el ETL (`CODIGO_INEI + PERIODO_ESTANDARIZADO + CODIGO_SIU_PROGRAMA + CODIGO_GRUPO_1 + CODIGO_GRUPO_3 + CODIGO_LOCAL + GUID_PERSONA`) y re-ejecutar el conteo de duplicados bajo esa clave ampliada antes de decidir la estrategia de deduplicación final. Esto queda marcado como **pendiente de validación** — no se debe asumir sin volver a correr el diagnóstico.

## 4. Problemas de calidad encontrados

| Dataset | Problema | Impacto | Tratamiento propuesto |
|---|---|---|---|
| Ingresantes | Nulos en `NACIONALIDAD` y `DEPARTAMENTO_NACIMIENTO` | Bajo (<1.5%) | Imputar `"No especificado"`; no eliminar filas (no son críticas para las preguntas del caso) |
| Ingresantes | Nulos masivos en columnas de discapacidad | ~99.6% | Dejar como `NaN`; no se usan en agregados para el dashboard |
| Matriculados | Duplicados exactos (902K) | ~4.94% | `drop_duplicates()` completo, conservando la primera ocurrencia |
| Matriculados | Duplicados bajo clave candidata (75.9K) | ~0.4% remanente | **Pendiente** — depende de si se amplía la clave con periodo (punto 3) |
| Matriculados | Nulos en `CODIGO_GRUPO_1/3` y `NOMBRE_GRUPO_1/3` (~3.5M, 19.6%) | Medio | Imputar `-1` / `"Sin agrupación"` — probablemente corresponde a programas sin sub-agrupación curricular, no a error de captura |
| Matriculados | Nulos en `ANIO_PERIODO_INGRESO` (~219K, 1.2%) | Bajo-medio | No mencionado en el informe original — **agregar como pendiente**: definir tratamiento (imputación vs. exclusión) antes del ETL |
| Institucional | Comas internas sin escapar | Inhabilita parseo (99.98% de líneas afectadas) | Descartar del pipeline; documentar exclusión |
| Ambos | Encoding `latin-1` y separador `|` | Bajo si se maneja explícitamente | Fijar `encoding="latin-1"` y `sep="|"` en la config de carga del ETL, no dejarlo a detección automática |

## 5. Preguntas que todavía no podemos responder

- **Relación ingresantes ↔ matriculados:** no existe clave directa entre ambas tablas (no comparten un ID de evento común). La única vía es agregar ambas por `(universidad, programa, periodo)` y comparar series — esto es suficiente para las preguntas obligatorias del caso, pero no permite trazar la trayectoria individual de un ingresante hacia su matrícula.
- **Granularidad definitiva de matriculados:** sigue sin resolverse (ver punto 3) hasta no probar la clave ampliada con periodo y confirmar si los 75,907 duplicados remanentes son duplicados reales de carga o corresponden a inscripciones legítimas en múltiples cursos/grupos que el esquema actual no distingue completamente.
- **Docentes:** dataset aún no explorado; se necesita repetir el mismo proceso de diagnóstico (esquema, calidad, clave) antes de poder usarlo para la pregunta adicional 3.3 si esta involucra docentes.
- **Institucional.csv:** queda sin resolver hasta contar con el catálogo INEI-2022 como referencia de desambiguación.

## 6. Qué debemos resolver antes de escribir `etl_pipeline.py`

1. **Re-evaluar la clave candidata de matriculados incluyendo `PERIODO_ESTANDARIZADO`** y volver a medir duplicados — esto es bloqueante para decidir la estrategia de deduplicación correcta (punto 3).
2. Definir tratamiento explícito para los nulos de `ANIO_PERIODO_INGRESO` (no cubierto en el informe original).
3. Confirmar la estrategia de deduplicación final para matriculados: ¿eliminar solo exactos y advertir sobre los remanentes de clave, o deduplicar agresivamente por clave ampliada?
4. Fijar `encoding` y `separador` explícitamente en la configuración de lectura (no depender de inferencia automática por archivo).
5. Definir el criterio de `Region_Sur` (qué departamentos entran) antes de calcularlo, ya que es un filtro obligatorio del caso (nivel sur del país).
6. Confirmar que `institucional.csv` queda formalmente fuera de alcance del pipeline v1, documentado como limitación en la PPT técnica.

---

**Nota de transparencia:** el punto más relevante de esta sección es el hallazgo sobre la clave de `matriculados` (punto 3) — el informe original lo reporta como "clave no perfecta, pendiente de validación de negocio", pero no señala que la ausencia del periodo en la clave es la causa estructural más probable, dado que el dataset es explícitamente multi-periodo. Se recomienda no avanzar con `etl_pipeline.py` hasta cerrar ese punto, porque afecta directamente la corrección de las series temporales que pide el dashboard.